In [1]:
# ============================================================
# LangChain create_agent with:
# - Tool calls
# - Tool-call metrics
# - Token metrics
# - Agent iteration metrics
# - Final structured Pydantic response
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import List, Optional, Any, Dict
from collections import Counter

# ============================================================
# 1. Structured final response schema
# ============================================================

class AutonomousAgentResponse(BaseModel):
    """
    This is the final structured response returned by the agent.

    Important:
    - This does not replace result["messages"].
    - Tool-call metrics still come from result["messages"].
    - This is your validated final business output.
    """

    final_answer: str = Field(
        description="The final response to the user's request."
    )

    task_completed: bool = Field(
        description="Whether the agent completed the user's request."
    )

    reasoning_summary: str = Field(
        description="Brief user-facing summary of the steps the agent took. Do not include hidden chain-of-thought."
    )

    tools_used: List[str] = Field(
        description="Names of tools used by the agent during the run."
    )

    key_findings: List[str] = Field(
        description="Important facts, observations, or tool outputs used to support the final answer."
    )

    limitations: List[str] = Field(
        description="Any limitations, missing information, or uncertainty."
    )

    recommended_next_steps: List[str] = Field(
        description="Suggested next steps for the user, if any."
    )

    confidence: float = Field(
        description="Confidence score from 0.0 to 1.0."
    )


# ============================================================
# 2. Content helpers
# ============================================================

def normalize_content(content: Any) -> str:
    """
    Handles plain string content and list-of-blocks content.
    LangChain message content can be a string or structured blocks.
    """
    if content is None:
        return ""

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []

        for block in content:
            if isinstance(block, str):
                parts.append(block)

            elif isinstance(block, dict):
                if "text" in block:
                    parts.append(str(block["text"]))
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))

            else:
                parts.append(str(block))

        return "\n".join(parts)

    return str(content)


def get_final_answer_from_messages(result: dict) -> str:
    """
    Returns the final non-empty AIMessage content.

    Note:
    When using structured output, the final AIMessage.content may sometimes
    be empty or less useful depending on provider strategy. In that case,
    prefer result["structured_response"].final_answer for the final answer.
    """
    for msg in reversed(result.get("messages", [])):
        if isinstance(msg, AIMessage):
            text = normalize_content(msg.content).strip()
            if text:
                return text

    return ""


def get_structured_response(result: dict) -> Optional[AutonomousAgentResponse]:
    """
    Returns the final Pydantic structured response if LangChain produced one.
    """
    return result.get("structured_response")


def get_final_answer(result: dict) -> str:
    """
    Preferred final-answer accessor.

    Uses structured_response.final_answer when available.
    Falls back to the final AIMessage content.
    """
    structured = get_structured_response(result)

    if structured is not None and hasattr(structured, "final_answer"):
        return structured.final_answer

    return get_final_answer_from_messages(result)


# ============================================================
# 3. Metrics helpers
# ============================================================

def summarize_agent_metrics(result: dict) -> dict:
    """
    Summarize agent run metrics while preserving tool-call visibility.

    This function intentionally reads from result["messages"].
    Do not switch this to structured_response, or you will lose tool-call metrics.
    """
    messages = result.get("messages", [])

    ai_messages = [m for m in messages if isinstance(m, AIMessage)]
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    human_messages = [m for m in messages if isinstance(m, HumanMessage)]

    tool_calls = []

    token_totals = {
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
    }

    model_calls_with_usage = 0

    token_usage_by_step = []

    ai_step = 0

    for message_index, msg in enumerate(messages):
        if not isinstance(msg, AIMessage):
            continue

        ai_step += 1

        # ----------------------------------------------------
        # Tool calls requested by the model
        # ----------------------------------------------------
        msg_tool_calls = getattr(msg, "tool_calls", None) or []

        for call in msg_tool_calls:
            tool_calls.append({
                "name": call.get("name"),
                "args": call.get("args"),
                "id": call.get("id"),
                "message_index": message_index,
                "ai_step": ai_step,
            })

        # ----------------------------------------------------
        # Token usage
        # ----------------------------------------------------
        usage = getattr(msg, "usage_metadata", None)

        step_input_tokens = 0
        step_output_tokens = 0
        step_total_tokens = 0

        if usage:
            model_calls_with_usage += 1

            step_input_tokens = usage.get("input_tokens", 0) or 0
            step_output_tokens = usage.get("output_tokens", 0) or 0
            step_total_tokens = usage.get("total_tokens", 0) or 0

            token_totals["input_tokens"] += step_input_tokens
            token_totals["output_tokens"] += step_output_tokens
            token_totals["total_tokens"] += step_total_tokens

        token_usage_by_step.append({
            "message_index": message_index,
            "ai_step": ai_step,
            "input_tokens": step_input_tokens,
            "output_tokens": step_output_tokens,
            "total_tokens": step_total_tokens,
            "tool_call_count": len(msg_tool_calls),
            "has_tool_calls": len(msg_tool_calls) > 0,
        })

    tool_name_counts = Counter(call["name"] for call in tool_calls)

    structured = get_structured_response(result)

    if structured is not None:
        try:
            structured_response_dict = structured.model_dump()
        except Exception:
            structured_response_dict = str(structured)
    else:
        structured_response_dict = None

    return {
        # Final answer
        "final_answer": get_final_answer(result),
        "final_answer_from_messages": get_final_answer_from_messages(result),

        # Structured response
        "has_structured_response": structured is not None,
        "structured_response_type": type(structured).__name__ if structured is not None else None,
        "structured_response": structured,
        "structured_response_dict": structured_response_dict,

        # Message counts
        "message_count": len(messages),
        "human_message_count": len(human_messages),
        "ai_message_count": len(ai_messages),
        "tool_message_count": len(tool_messages),

        # Agent loop metrics
        "agent_iterations": len(ai_messages),

        # Tool metrics
        "tool_call_count": len(tool_calls),
        "tool_name_counts": dict(tool_name_counts),
        "tool_calls": tool_calls,
        "tool_call_sequence": [call["name"] for call in tool_calls],
        "unique_tools_used": len(tool_name_counts),

        # Token metrics
        "model_calls_with_usage": model_calls_with_usage,
        "token_usage_by_step": token_usage_by_step,
        **token_totals,
    }


def get_tool_outputs(result: dict) -> List[dict]:
    """
    Extract tool outputs from ToolMessage objects.
    """
    outputs = []

    for message_index, msg in enumerate(result.get("messages", [])):
        if isinstance(msg, ToolMessage):
            content = normalize_content(msg.content)

            outputs.append({
                "message_index": message_index,
                "tool_name": getattr(msg, "name", None),
                "tool_call_id": getattr(msg, "tool_call_id", None),
                "output": content,
                "output_chars": len(content),
                "is_empty": len(content.strip()) == 0,
            })

    return outputs


def print_agent_report(result: dict) -> None:
    """
    Print the same report style you already have, plus structured response.
    """
    metrics = summarize_agent_metrics(result)

    print("Final answer:")
    print(metrics["final_answer"])

    print("\nFinal answer from messages:")
    print(metrics["final_answer_from_messages"])

    print("\nRun metrics:")
    for k, v in metrics.items():
        if k not in [
            "final_answer",
            "final_answer_from_messages",
            "tool_calls",
            "structured_response",
            "structured_response_dict",
            "token_usage_by_step",
        ]:
            print(f"{k}: {v}")

    print("\nTool calls:")
    for call in metrics["tool_calls"]:
        print(f"- ai_step={call['ai_step']} | {call['name']}: {call['args']}")

    print("\nToken usage by step:")
    for step in metrics["token_usage_by_step"]:
        print(
            f"- ai_step={step['ai_step']} | "
            f"input={step['input_tokens']} | "
            f"output={step['output_tokens']} | "
            f"total={step['total_tokens']} | "
            f"tool_calls={step['tool_call_count']}"
        )

    print("\nTool outputs:")
    for output in get_tool_outputs(result):
        print(
            f"- message_index={output['message_index']} | "
            f"tool_name={output['tool_name']} | "
            f"chars={output['output_chars']} | "
            f"preview={output['output'][:250]}"
        )

    print("\nStructured response:")
    structured = metrics["structured_response"]

    if structured is None:
        print("No structured_response found.")
    else:
        print(structured)

        print("\nStructured response JSON:")
        print(structured.model_dump_json(indent=2))

In [3]:
# ============================================================
# 4. Tool definitions
# ============================================================

DEMO_SALES_SCHEMA = """Demo sales CSV: examples/demo_data/sales.csv
Rows: 10,000
Columns:
- SalesOrderNumber: order id
- SalesOrderLineNumber: line item number
- OrderDate: order date
- CustomerName: customer name
- EmailAddress: customer email
- Item: product name
- Quantity: units sold
- UnitPrice: price per unit
- TaxAmount: tax charged
"""

DEMO_TOP_REVENUE_PRODUCTS = """Top products by estimated revenue in the demo sales CSV:
1. Road-150 Red, 48 — 337 units — $1,205,876.99
2. Road-150 Red, 62 — 336 units — $1,202,298.72
3. Road-150 Red, 52 — 302 units — $1,080,637.54
4. Road-150 Red, 56 — 295 units — $1,055,589.65
5. Road-150 Red, 44 — 281 units — $1,005,493.87
"""

DEMO_TOTAL_REVENUE = """Demo sales CSV summary:
- Rows: 10,000
- Estimated total revenue: $14,432,081.12
- Highest revenue product variant: Road-150 Red, 48
"""

SDD_DOC_SUMMARY = """Spec-Driven Development means steering AI coding agents with durable markdown specs instead of relying only on vibe coding and long chat history.
In this workflow, the constitution, feature specs, validation files, roadmap, and changelog keep the agent focused, reduce drift, and make the project easier to explain and reproduce.
SDD is part of harness engineering: the developer directs intent, boundaries, acceptance criteria, and validation while the agent handles more of the implementation work.
"""

WEB_SEARCH_RESULTS = {
    "latest on ai": "Search result: OpenCode is being discussed as a free alternative to Claude Code. Demo takeaway: the AI coding tool space is moving quickly, so durable specs help teams avoid rebuilding around every new tool trend.",
    "langchain tools": "Search result: LangChain tools let agents call Python functions with typed inputs. Demo takeaway: tools give an agent controlled ways to use external context, calculations, and actions.",
    "spec driven development": "Search result: Spec-driven workflows use written requirements, plans, and validation criteria to guide AI-assisted software delivery. Demo takeaway: specs help reduce drift and make agent work reviewable.",
    "streamlit fastapi demo": "Search result: Streamlit is commonly used for quick Python UIs, while FastAPI is used for JSON APIs. Demo takeaway: this repo separates the teaching UI from the agent backend.",
}


@tool
def run_sql(query: str) -> str:
    """Simulate a read-only SQL tool over the demo sales CSV.

    This classroom version does not execute SQL. It returns curated sales-data
    facts so students can see how an agent might call a database tool without
    adding a real database or arbitrary code execution to the notebook.
    """
    normalized = query.lower().strip()
    blocked_terms = ["insert", "update", "delete", "drop", "alter", "create", "attach"]

    if any(term in normalized for term in blocked_terms):
        return "Rejected: this demo SQL tool is read-only and does not run write or schema-changing statements."

    if "schema" in normalized or "column" in normalized or "table" in normalized:
        return DEMO_SALES_SCHEMA

    if "top" in normalized and "revenue" in normalized:
        return DEMO_TOP_REVENUE_PRODUCTS

    if "total" in normalized and "revenue" in normalized:
        return DEMO_TOTAL_REVENUE

    return (
        "Demo SQL tool received the query but did not execute it. "
        "Try asking for the sales schema, total revenue, or top products by revenue.\n\n"
        f"Query received: {query}"
    )


@tool
def validate_answer(answer: str, evidence: str) -> str:
    """Critique the answer for missing evidence, bad assumptions, or calculation errors."""
    if not answer.strip():
        return "Validation failed: answer is empty."

    if not evidence.strip():
        return "Validation warning: evidence is empty or limited."

    answer_terms = set(answer.lower().split())
    evidence_terms = set(evidence.lower().split())
    overlap = answer_terms.intersection(evidence_terms)

    if len(overlap) < 3:
        return "Validation warning: the answer has limited overlap with the provided evidence."

    return "Validation passed: the answer is supported by the provided evidence."


@tool
def search_docs(query: str) -> str:
    """Search internal documents for relevant context about SDD.

    This classroom version returns one concise SDD knowledge-base passage when
    the query asks about SDD. It intentionally avoids embeddings or file IO.
    """
    if "sdd" not in query.lower() and "spec" not in query.lower():
        return f"No internal documents found for query: {query}"

    return SDD_DOC_SUMMARY


@tool
def save_artifact(name: str, content: str) -> str:
    """Simulate saving an artifact and return a receipt without writing files."""
    safe_name = name.strip() or "untitled-artifact.md"
    preview = content.strip().replace("\n", " ")[:160]

    return (
        f"Artifact save simulated: {safe_name}\n"
        f"Characters: {len(content)}\n"
        f"Preview: {preview}"
    )


@tool
def weather(location: str) -> str:
    """Get deterministic demo weather for a location."""
    normalized = location.lower().strip()

    weather_by_location = {
        "new york": "The current weather in New York is sunny, 75°F.",
        "delano": "The current weather in Delano is foggy, 60°F.",
        "minneapolis": "The current weather in Minneapolis is cloudy, 48°F.",
        "phoenix": "The current weather in Phoenix is hot, 98°F.",
    }

    if normalized in weather_by_location:
        return weather_by_location[normalized]

    return f"Sorry, I don't have weather data for {location}."


@tool
def web_search(query: str) -> str:
    """Search a curated fake web index for classroom-safe demo results."""
    normalized = query.lower().strip()

    if normalized in WEB_SEARCH_RESULTS:
        return WEB_SEARCH_RESULTS[normalized]

    for topic, result in WEB_SEARCH_RESULTS.items():
        if topic in normalized or normalized in topic:
            return result

    return f"Sorry, I don't have web search capabilities for the query: {query}"



In [4]:

# ============================================================
# 5. LLM setup
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,
    project="zen-general-377713",
    location="global",
)


# ============================================================
# 6. Agent setup with structured response
# ============================================================

agent = create_agent(
    model=llm,
    tools=[
        run_sql,
        validate_answer,
        search_docs,
        save_artifact,
        weather,
        web_search,
    ],
    response_format=AutonomousAgentResponse,
    system_prompt="""
    You are an autonomous agent running a classroom demo.

    You may use tools repeatedly when needed. The available tools are deterministic teaching examples, not production integrations.

    Before finalizing:
    - Make sure the answer is grounded in tool results when tools were used.
    - Do not fabricate tool results.
    - Stop when the answer is complete and verified.

    Your final answer must be returned using the required structured response schema.

    In the structured response:
    - final_answer should directly answer the user's request.
    - tools_used should list the actual tools used.
    - key_findings should summarize important facts from tool outputs.
    - limitations should mention missing information or uncertainty.
    - confidence should be between 0.0 and 1.0.
    """
)


# ============================================================
# 7. Invoke the agent
# ============================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Use internal docs to explain SDD, check whether Delano is hotter than 50 degrees, search the web for latest on AI if it is, use the sales tool to identify the top revenue product, validate the answer, and save a simulated artifact named sdd-demo-summary.md.",
            }
        ]
    },
    config={
        # Optional, but useful for autonomous agents.
        # This is not exactly "max tool calls"; it is the graph recursion cap.
        "recursion_limit": 50
    },
)


# ============================================================
# 8. Inspect raw messages if desired
# ============================================================

print("Raw messages:")
for msg in result["messages"]:
    print(msg)
    print("-" * 100)


# ============================================================
# 9. Print full report
# ============================================================

print_agent_report(result)



Raw messages:
content='Use internal docs to explain SDD, check whether Delano is hotter than 50 degrees, search the web for latest on AI if it is, use the sales tool to identify the top revenue product, validate the answer, and save a simulated artifact named sdd-demo-summary.md.' additional_kwargs={} response_metadata={} id='91adefea-64e9-40c5-8337-e45ff087c266'
----------------------------------------------------------------------------------------------------
content=[] additional_kwargs={'function_call': {'name': 'search_docs', 'arguments': '{"query": "SDD"}'}, '__gemini_function_call_thought_signatures__': {'6940df2d-96ef-44fb-b9d7-4089a821c00b': 'AY89a19eP0KEMfIDRYculOuyvSajIqwcv0AQ63AMZXg2WbtgTiy/xhDtWpgskKjcWLf7ZHL/TsTwWeCjLca7aMujrSERnYCt3ioffmc5yIkhvUyoakW/oxqs7CEUKYmbYzxN0UQskiKEkYWdmSDO6B9hBvKR72h2G6NUN/VJg5Y2EUG4kKJ+5WPWrYENobrj+mWS9f00sNbqYwhShvvY9VGyA7MNuM43288jYPZqJS4fHGaPcn76FbtOUgODgsrUEzl6FNG+4ddUQqGeRKVZQYB6JOLNDb+p6H2dfALF5iRQPtL2UL2xJgIQz2j4ryIgqtMx70PIQBp4lz0aJIZ

In [5]:

# ============================================================
# 10. Programmatic access
# ============================================================

metrics = summarize_agent_metrics(result)
structured = get_structured_response(result)

# Tool-call metrics still available:
tool_call_count = metrics["tool_call_count"]
tool_name_counts = metrics["tool_name_counts"]
tool_calls = metrics["tool_calls"]
agent_iterations = metrics["agent_iterations"]
input_tokens = metrics["input_tokens"]
output_tokens = metrics["output_tokens"]
total_tokens = metrics["total_tokens"]

# Structured final response also available:
if structured is not None:
    final_structured_answer = structured.final_answer
    structured_dict = structured.model_dump()
else:
    final_structured_answer = None
    structured_dict = None

print("\nProgrammatic values:")
print("tool_call_count:", tool_call_count)
print("tool_name_counts:", tool_name_counts)
print("agent_iterations:", agent_iterations)
print("input_tokens:", input_tokens)
print("output_tokens:", output_tokens)
print("total_tokens:", total_tokens)
print("final_structured_answer:", final_structured_answer)
print("structured_dict:", structured_dict)


Programmatic values:
tool_call_count: 6
tool_name_counts: {'search_docs': 1, 'weather': 1, 'web_search': 1, 'run_sql': 1, 'validate_answer': 1, 'save_artifact': 1}
agent_iterations: 7
input_tokens: 12056
output_tokens: 2207
total_tokens: 14263
final_structured_answer: According to the internal documents, Spec-Driven Development (SDD) is a developer-directed workflow that guides AI coding agents using durable markdown specs (like feature specs, roadmaps, and validation files) instead of depending on volatile chat histories and 'vibe coding.' This harness engineering approach keeps development aligned and structured.

Delano is currently at 60°F (foggy), which is hotter than 50°F. Since this condition was met, a web search was conducted, showing that OpenCode is currently being discussed as a free alternative to Claude Code, illustrating how quickly AI tools are evolving and why durable specs (SDD) help teams stay grounded.

From the database sales tool, the top-performing product varia